In [1]:
# Modeling the Lotka-Volterra (+ diffusion) approach to linguistic evolution
import numpy as np
import math
import pandas as pd

In [2]:
# Helpers
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    R = 6371.0
    distance = R * c

    return distance

def solve_de(L, c, u, v, beta, gamma, dt):
    """
    Solves the differential equation, returning (du, dv) to be added to (u, v)
    L: The Laplacian (n x n)
    c: The competition coefficient (positive means u is higher status, negative means v) (1 x n)
    u: The concentrations of Galician speakers (n x 1)
    v: The concentrations of Spanish speakers (n x 1)
    beta: The diffusion coefficients (1 x n)
    gamma: The cross-diffusion coefficients for u (1 x n)
    dt: We're solving for du: du/dt --> du = dt(..)
    """

    # Lotka-Volterra Components
    du_lv = (c * u * v) + u * (1-u)
    dv_lv = (-c * u * v) + v * (1-v)

    # Diffusion Component
    du_diff = np.matmul(L, u) * beta + np.matmul(L, (u*v)) * gamma
    dv_diff = np.matmul(L, v) * beta + np.matmul(L, (u*v)) * (1 - gamma)

    # Final Result
    du = dt * (du_lv + du_diff)
    dv = dt * (dv_lv + dv_diff)

    return du, dv

In [3]:
# Read in data
df = pd.read_csv("data/galicia_filtered_census.csv")

In [4]:
# Define initial conditions and coefficients
u0, v0 = [], []
gamma = []
c = []
gamma = []

gamma_const = 1.
c_const = 0.75
high_init_const = (1 + c_const) / (1 + c_const ** 2)
low_init_const = (1 - c_const) / (1 + c_const ** 2)

avg_status = np.mean(df["status"])

# Init by status
for _, row in df.iterrows():
    # Higher status --> more galician
    if row["status"] > avg_status:
        u0.append(high_init_const)
        v0.append(low_init_const)
        c.append(c_const)
        gamma.append(gamma_const)
    else:
        u0.append(low_init_const)
        v0.append(high_init_const)
        c.append(-c_const)
        gamma.append(0)

# Init randomly
# TODO

In [16]:
# Define coords, Laplacian
n = df.shape[0]
coords = np.array([df['lat'], df['lon']]).T
populations = np.array(df['pop'])
W = np.zeros((n, n))

# Laplacian at diagonals should be 0...probably.
# Because it represents the weight of diffusion and between a municipality and 
# itself there shouldn't be any diffusion

# NOTE: There is a very big city which is a huge outlier in the Laplacian...could be an issue?

for i in range(0, n):
    for j in range(i+1, n):
        weight = (populations[i] * populations[j]) / (haversine(coords[i][0], coords[i][1], coords[j][0], coords[j][1]) ** 2)
        W[i][j] = weight
        W[j][i] = weight

DD = np.diag(np.sum(W, axis=1))
laplacian = W - DD

8.766065711652262
244810 12098
